# Week 6 Exercise: Frontier Return Reason Classifier (OpenRouter)

This version uses OpenRouter for frontier-model evaluation with base vs adapted (few-shot) prompts, using a fixed evaluation set.

In [ ]:
#!uv add openai datasets pandas scikit-learn

In [ ]:
import os
import re
import random
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from openai import OpenAI

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY', '')
client = OpenAI(base_url='https://openrouter.ai/api/v1', api_key=OPENROUTER_API_KEY) if OPENROUTER_API_KEY else None

BASE_MODEL = 'openai/gpt-4o-mini'
EVAL_SIZE = 80
RUN_BASE_EVAL = True
RUN_ADAPTED_EVAL = True

SYSTEM_PROMPT = (
    'You classify customer return reasons. '
    'Return exactly one label from: defective, wrong_item, size_or_fit, late_delivery, not_as_described, changed_mind.'
)
LABELS = ['defective', 'wrong_item', 'size_or_fit', 'late_delivery', 'not_as_described', 'changed_mind']

print('openrouter_key_set:', bool(OPENROUTER_API_KEY))


## Build Dataset

In [ ]:
def normalize_text(x: str) -> str:
    x = x.lower().strip()
    x = re.sub(r'\s+', ' ', x)
    return x


def assign_return_reason(text: str, stars: int):
    t = normalize_text(text)
    if any(k in t for k in ['broken', 'defect', 'stopped working', 'damaged', 'does not work', "didn't work"]):
        return 'defective'
    if any(k in t for k in ['wrong item', 'wrong product', 'received different', 'not what i ordered', 'different item']):
        return 'wrong_item'
    if any(k in t for k in ['too small', 'too big', 'does not fit', "didn't fit", 'size was off']):
        return 'size_or_fit'
    if any(k in t for k in ['arrived late', 'late delivery', 'delayed shipping', 'shipping took too long', 'expected date']):
        return 'late_delivery'
    if any(k in t for k in ['not as described', 'misleading', 'looked different', 'description was wrong', 'nothing like the photo']):
        return 'not_as_described'
    if stars <= 2 and any(k in t for k in ['returned', 'returning', 'refund', 'not worth it', 'changed my mind']):
        return 'changed_mind'
    return None

try:
    ds = load_dataset('amazon_reviews_multi', 'en', split='train[:16000]', trust_remote_code=True)
    raw_df = ds.to_pandas()[['review_title', 'review_body', 'stars']].dropna().copy()
    raw_df['text'] = (raw_df['review_title'].fillna('') + '. ' + raw_df['review_body'].fillna('')).str.strip()
except Exception:
    ds = load_dataset('yelp_review_full', split='train[:16000]')
    raw_df = ds.to_pandas()[['text', 'label']].dropna().copy()
    raw_df['stars'] = raw_df['label'].astype(int) + 1

raw_df = raw_df[raw_df['text'].str.len() > 30].reset_index(drop=True)
raw_df['label'] = [assign_return_reason(t, int(s)) for t, s in zip(raw_df['text'], raw_df['stars'])]
df = raw_df[raw_df['label'].isin(LABELS)].copy().reset_index(drop=True)

counts = df['label'].value_counts()
df = df[df['label'].isin(counts[counts >= 5].index)].copy().reset_index(drop=True)

train_df, temp_df = train_test_split(df, test_size=0.3, random_state=SEED, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED, stratify=temp_df['label'])

print('train:', len(train_df), 'val:', len(val_df), 'test:', len(test_df))
print(df['label'].value_counts())


## Prepare JSONL Artifacts

In [ ]:
import json
def training_record(row):
    return {
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': row['text']},
            {'role': 'assistant', 'content': row['label']}
        ]
    }


def write_jsonl(frame, path):
    with open(path, 'w') as f:
        for _, r in frame.iterrows():
            f.write(json.dumps(training_record(r), ensure_ascii=False) + '\n')

out_dir = Path('jsonl')
out_dir.mkdir(exist_ok=True)
train_path = out_dir / 'week6_returns_train.jsonl'
val_path = out_dir / 'week6_returns_val.jsonl'

write_jsonl(train_df, train_path)
write_jsonl(val_df, val_path)

print('wrote', train_path, len(train_df))
print('wrote', val_path, len(val_df))


## Fixed Evaluation Set

In [ ]:
eval_df = test_df[['text', 'label']].sample(n=min(EVAL_SIZE, len(test_df)), random_state=SEED).reset_index(drop=True)

demo_fewshot = train_df[['text', 'label']].sample(n=min(6, len(train_df)), random_state=SEED).to_dict('records')
eval_df.head(3)


In [ ]:
def extract_label(x: str):
    t = normalize_text(x)
    for lbl in LABELS:
        if lbl in t or lbl.replace('_', ' ') in t:
            return lbl
    return 'unknown'


def classify_with_openrouter(text: str, model_name: str, fewshot=None):
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    if fewshot:
        for ex in fewshot:
            messages.append({'role': 'user', 'content': ex['text']})
            messages.append({'role': 'assistant', 'content': ex['label']})
    messages.append({'role': 'user', 'content': text})

    resp = client.chat.completions.create(
        model=model_name,
        messages=messages,
        temperature=0,
        extra_headers={
            'HTTP-Referer': 'https://github.com/davidkamere/llm_engineering',
            'X-Title': 'week6-frontier-classifier'
        }
    )
    raw = (resp.choices[0].message.content or '').strip()
    return extract_label(raw), raw


def evaluate_model(model_name: str, fewshot=None):
    preds = []
    raws = []
    for txt in eval_df['text'].tolist():
        p, raw = classify_with_openrouter(txt, model_name, fewshot=fewshot)
        preds.append(p)
        raws.append(raw)
    y_true = eval_df['label'].tolist()
    acc = accuracy_score(y_true, preds)
    f1 = f1_score(y_true, preds, average='macro')
    return preds, raws, acc, f1


## Base Frontier Evaluation

In [ ]:
if RUN_BASE_EVAL:
    if client is None:
        raise ValueError('OPENROUTER_API_KEY not set')
    base_preds, base_raws, base_acc, base_f1 = evaluate_model(BASE_MODEL, fewshot=None)
    print('base_accuracy:', round(base_acc, 4))
    print('base_macro_f1:', round(base_f1, 4))
    print(classification_report(eval_df['label'].tolist(), base_preds, digits=4, zero_division=0))
else:
    print('Set RUN_BASE_EVAL=True to run base evaluation')


## Adapted Frontier Evaluation (Few-Shot Prompt Tuning)

In [ ]:
if RUN_ADAPTED_EVAL:
    if client is None:
        raise ValueError('OPENROUTER_API_KEY not set')
    adapted_preds, adapted_raws, adapted_acc, adapted_f1 = evaluate_model(BASE_MODEL, fewshot=demo_fewshot)
    print('adapted_accuracy:', round(adapted_acc, 4))
    print('adapted_macro_f1:', round(adapted_f1, 4))
    print(classification_report(eval_df['label'].tolist(), adapted_preds, digits=4, zero_division=0))
else:
    print('Set RUN_ADAPTED_EVAL=True to run adapted evaluation')


## Comparison Output

In [ ]:
rows = []
if 'base_acc' in locals():
    rows.append({'model_variant': 'base', 'model': BASE_MODEL, 'accuracy': round(base_acc, 4), 'macro_f1': round(base_f1, 4)})
if 'adapted_acc' in locals():
    rows.append({'model_variant': 'adapted_fewshot', 'model': BASE_MODEL, 'accuracy': round(adapted_acc, 4), 'macro_f1': round(adapted_f1, 4)})

results_df = pd.DataFrame(rows)
results_df


In [ ]:
if len(results_df):
    Path('artifacts').mkdir(exist_ok=True)
    results_df.to_csv('artifacts/week6_frontier_results.csv', index=False)
    eval_df.to_csv('artifacts/week6_eval_set.csv', index=False)
    print('saved artifacts/week6_frontier_results.csv')
    print('saved artifacts/week6_eval_set.csv')
else:
    print('No results yet to save')
